# NASA POWER: clima para México por serie USyV

Extracción masiva de parámetros climáticos desde el **datamart público de NASA POWER en AWS S3**
(stores Zarr del Analysis Ready Datastore), evitando la API REST y sus límites.

- **Fuente**: `https://nasa-power.s3.us-west-2.amazonaws.com/{store}/temporal/power_{store}_daily_temporal_{lst|utc}.zarr`
- **Stores usados**: `merra2` (meteorología, diario 1981→~2029, grilla 0.5°×0.625°) y `syn1deg` (solar, diario 2001→~2029, grilla 1°×1°).
- **Parámetros**: subset agrícola del diccionario `data/nasa-power-parameters.ods` (18 códigos; se excluyen SNODP, CLRSKY_DAYS e IMERG_* — este último existe en `imerg/` a 0.1° pero sólo en UTC y con mucho más peso).
- **Años por serie USyV** (mapeo de `docs/metodologia_usv_inegi_mapbiomas.md`): I→1981, II→1985, III→1993, IV→2002, V→2007, VI→2015, VII→2018.
- **Salida**: `data/interm/NASA/serie_{I..VII}/{param}.parquet` en formato largo (`time, lat, lon, value, units`).

Unidades: las del ARD ya vienen en unidades de usuario (°C, mm/día, W m⁻², fracción, degree-day, cm); se registran en la columna `units` y en `catalogo_parametros.csv`.

In [1]:
from scripts.nasa_power import (
    PARAMETROS_AGRICOLAS, SERIES_ANIOS, OUT_DIR,
    parsear_parametros, anios_disponibles, descargar_subset,
)
import pandas as pd

# Diccionario completo del .ods (18 parámetros)
diccionario = parsear_parametros()
diccionario

,code,name,description,kw1,kw2
0,T2M,Temperature at 2 Meters,The average air (dry bulb) temperature at 2 me...,"Dry Bulb Temperature at 2 Meters,Near-Surface ...","Dry Bulb Temperature at 2 Meters,Near-Surface ..."
1,T2M_MAX,Temperature at 2 Meters Maximum,The maximum hourly air (dry bulb) temperature ...,"Maximum Dry Bulb Temperature at 2 Meters,Maxim...","Maximum Dry Bulb Temperature at 2 Meters,Maxim..."
2,T2M_MIN,Temperature at 2 Meters Minimum,The minimum hourly air (dry bulb) temperature ...,"Minimum Dry Bulb Temperature at 2 Meters,Minim...","Minimum Dry Bulb Temperature at 2 Meters,Minim..."
3,PRECTOTCORR,Precipitation Corrected,The average MERRA-2 bias corrected total preci...,"Bias-Corrected Precipitation,Adjusted Total Pr...",
4,PW,Precipitable Water,The total atmospheric water vapor contained in...,"Total Column Water Vapor,Integrated Water Vapo...","Total Column Water Vapor,Integrated Water Vapo..."
5,GWETROOT,Root Zone Soil Wetness,The amount of water and water vapor available ...,"Root Zone Soil Saturation Fraction,Root Zone W...","Root Zone Soil Saturation Fraction,Root Zone W..."
6,ALLSKY_SFC_UV_INDEX,All Sky Surface UV Index,The ultraviolet radiation exposure index.,"UV Index (UVI),Global Solar UV Index","UV Index (UVI),Global Solar UV Index,Solar,Sol..."
7,CDD0,Cooling Degree Days Above 0 C,The daily accumulation of degrees above a 0-de...,"CDD Base 0 C,Cooling Degree Days Base 0,Coolin...","CDD Base 0 C,Cooling Degree Days Base 0,Coolin..."
8,EVPTRNS,Evapotranspiration Energy Flux,The evapotranspiration energy flux at the surf...,"Evapotranspiration Flux,Latent Heat Flux from ...","Evapotranspiration Flux,Latent Heat Flux from ..."
9,FROST_DAYS,Frost Days,A frost day occurs when the 2m temperature coo...,"Number of Frost Days,Days Below Freezing,Frost...","Number of Frost Days,Days Below Freezing,Frost..."


## Disponibilidad por serie

Los parámetros `syn1deg` (PW, UV_INDEX, SW_DIRH, PSH) sólo existen desde 2001,
por lo que las series I–III no los tienen. Los `merra2` cubren las 7 series.

In [2]:
disponible = pd.DataFrame(
    {p: anios_disponibles(p) for p in PARAMETROS_AGRICOLAS}
).T
# columnas = series; valores = año o NaN
print("Parámetros agrícolas y años descargados por serie:")
disponible

Parámetros agrícolas y años descargados por serie:


,I,II,III,IV,V,VI,VII
T2M,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0
T2M_MAX,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0
T2M_MIN,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0
PRECTOTCORR,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0
GWETROOT,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0
EVPTRNS,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0
TSURF,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0
FROST_DAYS,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0
CDD0,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0
HDD0,1981.0,1985.0,1993.0,2002.0,2007.0,2015.0,2018.0


## Descarga

`descargar_subset()` lee los chunks remotos de S3 bajo demanda (sólo bbox México ×
años objetivo), escribe un parquet por serie y deja `catalogo_parametros.csv`.
Es idempotente a nivel de ejecución pero **vuelve a pedir los chunks** si se
re-ejecuta; la descarga completa tarda varios minutos.

In [3]:
# Si los parquets ya existen (p. ej. corrida previa por CLI), no repetir.
import os
completo = all(
    (OUT_DIR / f"serie_{s}" / f"{p}.parquet").exists()
    for s in SERIES_ANIOS
    for p in PARAMETROS_AGRICOLAS
    if s in anios_disponibles(p)
)
if not completo:
    descargar_subset()
else:
    print("Salidas ya presentes en", OUT_DIR)

Salidas ya presentes en /home/dusty/Documents/Repositories/proyecto_agronomia_Q/data/interm/NASA


## Verificación

Comprobaciones básicas: dimensiones del parquet, rango de fechas, número de
celdas de la malla y plausibilidad de valores.

In [4]:
def resumen(param, serie):
    ruta = OUT_DIR / f"serie_{serie}" / f"{param}.parquet"
    df = pd.read_parquet(ruta)
    return pd.Series({
        "param": param, "serie": serie,
        "filas": len(df),
        "celdas": df[["lat", "lon"]].drop_duplicates().shape[0],
        "lat": (df.lat.min(), df.lat.max()),
        "lon": (df.lon.min(), df.lon.max()),
        "fecha_min": df.time.min().date(),
        "fecha_max": df.time.max().date(),
        "valor_min": df.value.min(),
        "valor_med": df.value.median(),
        "valor_max": df.value.max(),
        "nan_%": round(100 * df.value.isna().mean(), 2),
        "units": df.units.iloc[0],
    })

resumenes = [
    resumen(p, s)
    for p in ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "GWETROOT",
              "EVPTRNS", "TSURF", "FROST_DAYS", "CDD0", "HDD0"]
    for s in ["I", "VII"]
    if (OUT_DIR / f"serie_{s}" / f"{p}.parquet").exists()
]
pd.DataFrame(resumenes).set_index(["param", "serie"])

filas  celdas           lat                lon  \
param       serie                                                    
T2M         I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
T2M_MAX     I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
T2M_MIN     I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
PRECTOTCORR I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
GWETROOT    I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
EVPTRNS     I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
TSURF       I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
FROST_DAYS  I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
CDD0        I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
HDD0        I      754455    2067  (14.0, 33.0)  (-118.75, -86.25)   
            VII    754455    2067  (14.0, 33.0)  (-118.75, -86.25)   

                    fecha_min   fecha_max  valor_min  valor_med   valor_max  \
param       serie                                                             
T2M         I      1981-01-01  1981-12-31  -5.969971  23.450012   40.049988   
            VII    2018-01-01  2018-12-31  -7.750000  24.309998   41.359985   
T2M_MAX     I      1981-01-01  1981-12-31  -3.219971  25.950012   48.599976   
            VII    2018-01-01  2018-12-31  -4.859985  26.770020   48.919983   
T2M_MIN     I      1981-01-01  1981-12-31 -11.289978  21.270020   33.570007   
            VII    2018-01-01  2018-12-31 -11.440002  22.090027   33.919983   
PRECTOTCORR I      1981-01-01  1981-12-31   0.000000   0.190002  166.080017   
            VII    2018-01-01  2018-12-31   0.000000   0.169983  269.770020   
GWETROOT    I      1981-01-01  1981-12-31   0.219971   0.530029    1.000000   
            VII    2018-01-01  2018-12-31   0.200012   0.520020    1.000000   
EVPTRNS     I      1981-01-01  1981-12-31   0.000000   3.369995  184.979980   
            VII    2018-01-01  2018-12-31   0.000000   1.750000  187.909973   
TSURF       I      1981-01-01  1981-12-31  -5.429993  22.320007   43.789978   
            VII    2018-01-01  2018-12-31  -7.409973  23.429993   44.770020   
FROST_DAYS  I      1981-01-01  1981-12-31   0.000000   0.000000    1.000000   
            VII    2018-01-01  2018-12-31   0.000000   0.000000    1.000000   
CDD0        I      1981-01-01  1981-12-31   0.000000  23.630005   40.380005   
            VII    2018-01-01  2018-12-31   0.000000  24.479980   41.099976   
HDD0        I      1981-01-01  1981-12-31   0.000000   0.000000    5.400024   
            VII    2018-01-01  2018-12-31   0.000000   0.000000    7.789978   

                   nan_%         units  
param       serie                       
T2M         I       0.00             C  
            VII     0.00             C  
T2M_MAX     I       0.00             C  
            VII     0.00             C  
T2M_MIN     I       0.00             C  
            VII     0.00             C  
PRECTOTCORR I       0.00        mm/day  
            VII     0.00        mm/day  
GWETROOT    I      43.64             1  
            VII    43.64             1  
EVPTRNS     I      43.64         W m-2  
            VII    43.64         W m-2  
TSURF       I      43.64             C  
            VII    43.64             C  
FROST_DAYS  I       0.00          Days  
            VII     0.00          Days  
CD

In [5]:
# Parámetros syn1deg (sólo series IV-VII)
resumenes_sol = [
    resumen(p, "VII")
    for p in ["PW", "ALLSKY_SFC_UV_INDEX", "ALLSKY_SFC_SW_DIRH", "PSH"]
]
pd.DataFrame(resumenes_sol).set_index("param")

,serie,filas,celdas,lat,lon,fecha_min,fecha_max,valor_min,valor_med,valor_max,nan_%,units
param,,,,,,,,,,,,
PW,VII,228855,627,"(14.5, 32.5)","(-118.5, -86.5)",2018-01-01,2018-12-31,0.140015,3.070007,6.630005,0.0,cm
ALLSKY_SFC_UV_INDEX,VII,228855,627,"(14.5, 32.5)","(-118.5, -86.5)",2018-01-01,2018-12-31,0.099976,1.979980,3.840027,0.0,W m-2 x 40
ALLSKY_SFC_SW_DIRH,VII,228855,627,"(14.5, 32.5)","(-118.5, -86.5)",2018-01-01,2018-12-31,2.140015,151.390015,361.099976,0.0,W m-2
PSH,VII,228855,627,"(14.5, 32.5)","(-118.5, -86.5)",2018-01-01,2018-12-31,0.309998,5.590027,9.359985,0.0,W m-2


## Spot-check contra la API REST

Comparación de una celda concreta (centro de la Ciudad de México) entre el
parquet extraído del Zarr y un request puntual a la API `daily/point`.
Deben coincidir exactamente: misma celda de la malla nativa.

In [6]:
import requests

# Celda merra2 más cercana a la CDMX dentro del bbox
df = pd.read_parquet(OUT_DIR / "serie_VII" / "T2M.parquet")
celda = (df[["lat", "lon"]].drop_duplicates()
         .assign(d=lambda d: (d.lat - 19.43).abs() + (d.lon + 99.13).abs())
         .nsmallest(1, "d")[["lat", "lon"]].iloc[0])
print("Celda:", celda.to_dict())

params = {"start": "20180615", "end": "20180616",
          "latitude": float(celda.lat), "longitude": float(celda.lon),
          "community": "ag", "parameters": "T2M", "format": "JSON"}
resp = requests.get(
    "https://power.larc.nasa.gov/api/temporal/daily/point", params=params,
    timeout=60,
)
api = resp.json()["properties"]["parameter"]["T2M"]
print("API:", api)

local = df[(df.lat == celda.lat) & (df.lon == celda.lon) &
           (df.time == "2018-06-15")].value.iloc[0]
print("Zarr:", local)

Celda: {'lat': 19.5, 'lon': -99.375}


API: {'20180615': 14.94, '20180616': 15.03}
Zarr: 14.940002


## Salidas y siguientes pasos

- `data/interm/NASA/serie_{I..VII}/{param}.parquet` — formato largo por serie.
- `data/interm/NASA/catalogo_parametros.csv` — metadatos del subset.

Para cruzar con los puntos agrícolas (`data/process/INEGI/puntos_agricolas.*`,
EPSG:6372): asignar cada punto a la celda `lat/lon` más cercana (o interpolar)
y hacer join por `time`. Los parámetros `syn1deg` tienen grilla 1° (~111 km),
coarser que la de `merra2` (~50–60 km): tenerlo en cuenta al modelar.